# Lesson 4, Part 1

Python has a package **numpy** ("numerical Python") that 
makes repeated computations much easier to work with and far faster. 
Imported as **np** by convention, its heart is a special data type called **ndarray**, 
short for n-dimensional array. In some ways it resembles a list or list of lists.

In [ ]:
import numpy as np
# Any list is easily converted into an ndarray, which looks similar and retains its indexing properties
a_list = [0,1,2,4,8]
arr = np.array(a_list)
print('a_list =',a_list,'converts to arr =',arr,', which matches apart from spacing and commas')
print('arr has type',type(arr),', length is',len(arr))
print('arr[0]=', arr[0],', arr[1:4]=',arr[1:4],', arr[-1]=',arr[-1])

a1_list = ['dog','cat',3,6,'bird']
arr1 = np.array(a1_list)
print('\na1_list =',a1_list,'converts to arr1 =',arr1)   # \n inside string adds a new line

a2D_list = [[0,1,2,3],[4,5,6,7],[8,9,10,11]]
arr2D = np.array(a2D_list)
print('\na2D_list =',a2D_list,'converts to arr2D =\n',arr2D) 
print('arr2D length is',np.shape(arr2D),', arr2D[0]=', arr2D[0],', arr2D[1][1]=',arr2D[1][1])

Like lists, ndarrays and their slices typically pass values to other variables
through references, except for single elements and explicit full copies. 
Full copies for ndarrays are best made through *name*.**copy()**, although **np.copy**(*name*) usually works too.

In [ ]:
arr = np.array([0,1,2,3,4,5])
print('original value of arr is',arr)
a_num = arr[4]              # passes single value, not list
a_num = 17                  
arr1 = arr.copy()           # makes full copy
arr1[2] = 35                
arr2 = np.copy(arr[:-2])    # full copy except with ragged lists
arr2[-1] = 67               
print('arr is still',arr)   # not affected by previous changes  

a_slice = arr[3:]           # assigns new name to slice
a_slice[0] = -10
print('arr is now',arr)     # changing the slice changes arr

Unlike lists of lists, ndarrays want all rows to have the same length, all columns to have the same length, and so on. 

In [ ]:
# numpy doesn't like ragged lists with unequal row length
ragged_list = [[0,1,2,3],[4,5],[6,7,8]]
arr_ragged = np.array(ragged_list)
print('ragged_list =',ragged_list,'converts to')
print('arr_ragged =',arr_ragged)

Furthermore, numpy converts all numbers to the same data type: 
typically float64, the counterpart of standard Python float.

In [ ]:
a3_list = [0,1,2.5]
arr3 = np.array(a3_list)
print('\na3_list =',a3_list,'converts to arr3 =',arr3)
print('\ndata types in a3_list are',type(a3_list[0]),type(a3_list[1]),type(a3_list[2]))
print('\ndata types in arr3 are',type(arr3[0]),type(arr3[1]),type(arr3[2]))

The main reason for this regimentation is to keep all elements 
of the arrays regularly spaced in a continuous block. 
Regular spacing makes it easy for numpy to search for elements 
and apply similar operations to them. 
No **for** loop or **list comprehension** is needed. 
Arithmetic operators work element by element.

In [ ]:
a = np.array([0,1,2.5])
print(a,'+ 2  =',a+2)
print(a,'- 2  =',a+2)
print(a,'* 2  =',a*3)
print(a,'/ 2  =',a/2)
print(a,'** 2 =',a**2)

Given two ndarrays of the same dimensions, arithmetic operators work on each pair of elements sharing the same index.

In [ ]:
a = np.array([0,1,2.5])
b = np.array([2,3,4])

print(a,' + ',b,'  = ',a+b)
print(a,' - ',b,'  = ',a-b)
print(a,' * ',b,'  = ',a*b)
print(a,' / ',b,'  = ',a/b)
print(a,' ** ',b,' = ',a**b)

Logical operators also work element-by-element. We can use them to form 'masks' that extract the elements we're interested in.

In [ ]:
a2D_list = [[0,1,2,3],[4,5,6,7],[8,9,10,11]]
arr2D = np.array(a2D_list)

print('arr2D =\n',arr2D)
is_odd = (arr2D % 2 == 1)
print('\nis_odd =\n',is_odd)
odd_elements = arr2D[is_odd]
print('\nodd slements of arr2D are',odd_elements)

The command **np.arange** works like list(range), except that the starting value doesn't have to be an integer. 
The command **np.linspace** works like the linear_spacing function we built before.

In [ ]:
print(np.arange(3), np.arange(2,6), np.arange(3,20,5))
print(np.linspace(0,3,3), np.linspace(2,6,4), np.linspace(20,3,5))

Numpy commands are speedy in large arrays. 
To illustrate, we can import and use the command **time.time**. 
When lists lengthen tenfold, execution time for the list comprehension
increases roughly tenfold as well. 
Execution time for ndarrays rises more slowly
because numpy does many computations in parallel.

In [ ]:
import time

for num in (1E2,1E3,1E4,1E5):  # 1En = 1 followed by n zeros
    arr = np.linspace(0,1000,round(num))
    a_list = arr.tolist()    # this transforms an ndarray into a standard list
    print('array length =',round(num))
    start = time.time()
    for iter in range(200):
        arr*2+1
    end = time.time()
    time_ndarray = end - start
    print('100 iterations for nd_array take',time_ndarray,'seconds')
    start = time.time()
    for iter in range(100):
        [val*2+1 for val in a_list]
    end = time.time()
    time_list = end - start
    ratio = round(time_list / time_ndarray)
    print('100 iterations for list take',time_list,'seconds, which is',ratio,'times as long')

Let's use numpy to simplify polynomial calculations. 
We'll start by defining the polynomial as an ndarray of its coefficients from either a single coef list or dual coef/power lists. 
Each coef[i] is the coefficient for *x*<sup>power[i]</sup> if power is given and *x*<sup>i</sup> if it is not. 
All other coefficients are presumed zero. We'll use **np.zeros** to create an ndarray filled with zeros.

In [ ]:
# return coefficients of polynomial as ndarray
def coef_poly(coef,power=None):
    if power==None: 
        return np.array(coef)
    n = max(power)
    coef_full = np.zeros(n+1)   
    coef_full[power] = coef     # = [coef_full[power[i]] = coef[i] for i in range(len(power))]
    return coef_full

# demo
print(coef_poly([1,2,3]))       # = 1 + 2*x + 3*x^2
print(coef_poly([-1,1],[2,4]))  # = -x^2 + x^4

If we recast *x* as an ndarray, we don't need separate code for single numbers and lists,
since numpy will automatically evaluate functions for every value in *x*.

In [ ]:
def eval_poly(x,coef):
    poly = coef[-1]
    for c in coef[-2::-1]:
        poly = poly * x  + c
    return poly

x = np.linspace(1,4,4)   # don't need linear_spacing() since np.linspace() is built in
coef = coef_poly([1],[4])
y = eval_poly(x,coef)    # don't need polynomial_list() since numpy will evaluate for every element of x
print('x =',x,', x**4 =',y)

Let's refactor our derivative code to return the coefficients of the derivative, 
which we can evaluate at specific *x* as needed. 
Then let's repeat for our integral code.

In [ ]:
def deriv_poly(coef):
    if len(coef)==1: return [0]
    return coef[1:] * np.arange(1,len(coef))

# evaluate derivative at various x
def eval_deriv_poly(x,coef):
    coef_deriv = deriv_poly(coef)
    return eval_poly(x, coef_deriv)

# coefficients of integral of polynomial, ignoring the integration constant 
def integral_poly(coef):
    n = len(coef) + 1   
    coef_integral = np.zeros(n)
    coef_integral[1:] = coef / np.arange(1,n)  # element at [k+1] is coef[k]/(k+1)
    return coef_integral

# evaluate integral at various x
def eval_integral_poly(x,coef):
    coef_integral = integral_poly(coef)
    return eval_poly(x, coef_integral)

coef = coef_poly([1],[4])
coef_deriv = deriv_poly(coef)
coef_integral = integral_poly(coef)
print('Coefficients',coef,'for polynomial,',coef_deriv,'for derivative, \nand',coef_integral,'for integral')
x = np.array([2,3])  
y_deriv = eval_deriv_poly(x,coef)
print('Derivatives at x =',x,'are',y_deriv)
y_integral = eval_integral_poly(x,coef)
print('integrals at x =',x,'are',y_integral,'+ constant')

We can evaluate second and higher derivatives by taking derivatives of derivatives.

In [ ]:
coef = coef_poly([1],[4])
coef_deriv = deriv_poly(coef)
coef_integral = integral_poly(coef)
print('Coefficients',coef,'for polynomial,',coef_deriv,'for derivative, \nand',coef_integral,'for integral')
x = np.array([2,3])  
y_deriv = eval_deriv_poly(x,coef)
print('Derivatives at x =',x,'are',y_deriv)
y_integral = eval_integral_poly(x,coef)
print('integrals at x =',x,'are',y_integral,'+ constant')

Let's compare the exact second derivative with the ratio to *h*<sup>2</sup> 
of *f*(*x*+*h*) - 2*f*(*x*) + *f*(*x-h*), where *h* is small.

In [ ]:
def deriv_polynomial(coef, order=1):
    for i in range(order):  
        if len(coef)==1: return 0
        coef = coef[1:] * np.arange(1,len(coef))
    return coef

def eval_deriv_polynomial(x, coef, order=1):
    coef_deriv = deriv_polynomial(coef,order)
    return eval_poly(x, coef_deriv)
    
def compare_approx_second_deriv(xmin,xmax,coef,nudge=0.001):
    x = np.linspace(xmin,xmax,500)
    exact = eval_deriv_polynomial(x, coef, 2)
    numerator = eval_poly(x+nudge, coef) - 2 * eval_poly(x,coef) + eval_poly(x-nudge,coef)
    denominator = nudge * nudge
    approx = numerator / denominator
    abs_diff = np.abs(approx - exact)   # np.abs takes absolute value of each difference
    sum_abs_diff = np.sum(abs_diff)     # np.sum sums the absolute values
    sum_abs_exact = np.sum(np.abs(exact))
    avg_rel_abs_diff = sum_abs_diff / sum_abs_exact
    return avg_rel_abs_diff

xmin,xmax = 0,10
coef = np.array([1,-1,1,-1,1,-1])
difference = compare_approx_second_deriv(xmin,xmax,coef,0.001)
print("Average relative absolute difference =",difference)